# Что делает рецепт хорошим?
## Анализ популярности, пользы и реакции людей

Связываем три «правды» о рецепте и смотрим, совпадают ли они:

- 📈 **Популярность** — рейтинг и кол-во оценок
- 🥗 **Польза** — калории / нутриенты
- 💬 **Реакция** — текстовые отзывы

**Главный вопрос («вау»):** самые популярные рецепты — не самые здоровые, а в отзывах хвалят часто не за то, что заявлено в рецепте. Этот контраст и ищем.

## Источники данных

1. **Food.com Recipes and Interactions** (Kaggle, автор `shuyangli94`) — основа:
   - `RAW_recipes.csv` — рецепты (ингредиенты, время, шаги, колонка `nutrition`)
   - `RAW_interactions.csv` — взаимодействия (оценки + текстовые отзывы)
2. **USDA FoodData Central** (опционально) — более точные нутриенты, если понадобится уточнить.

In [ ]:
# Установка Kaggle CLI (разкомментируй и запусти один раз)
# %pip install kaggle

In [ ]:
# Скачивание датасета в day_2/data/ и распаковка (--unzip)
# Запускай после настройки kaggle.json. Размер ~0.3 GB.
# !kaggle datasets download -d shuyangli94/food-com-recipes-and-user-interactions -p data --unzip

## Шаг 1. Загрузка и знакомство с данными

In [1]:
import ast
from pathlib import Path

import numpy as np
import pandas as pd

DATA = Path("data")   # папка day_2/data/ с распакованными CSV

recipes = pd.read_csv(DATA / "RAW_recipes.csv")
interactions = pd.read_csv(DATA / "RAW_interactions.csv")

print("recipes:     ", recipes.shape)
print("interactions:", interactions.shape)
recipes.head()

recipes:      (231637, 12)
interactions: (1132367, 5)


,name,id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients
0,arriba baked winter squash mexican style,137739,55,47892,2005-09-16,"['60-minutes-or-less', 'time-to-make', 'course...","[51.5, 0.0, 13.0, 0.0, 2.0, 0.0, 4.0]",11,"['make a choice and proceed with recipe', 'dep...",autumn is my favorite time of year to cook! th...,"['winter squash', 'mexican seasoning', 'mixed ...",7
1,a bit different breakfast pizza,31490,30,26278,2002-06-17,"['30-minutes-or-less', 'time-to-make', 'course...","[173.4, 18.0, 0.0, 17.0, 22.0, 35.0, 1.0]",9,"['preheat oven to 425 degrees f', 'press dough...",this recipe calls for the crust to be prebaked...,"['prepared pizza crust', 'sausage patty', 'egg...",6
2,all in the kitchen chili,112140,130,196586,2005-02-25,"['time-to-make', 'course', 'preparation', 'mai...","[269.8, 22.0, 32.0, 48.0, 39.0, 27.0, 5.0]",6,"['brown ground beef in large pot', 'add choppe...",this modified version of 'mom's' chili was a h...,"['ground beef', 'yellow onions', 'diced tomato...",13
3,alouette potatoes,59389,45,68585,2003-04-14,"['60-minutes-or-less', 'time-to-make', 'course...","[368.1, 17.0, 10.0, 2.0, 14.0, 8.0, 20.0]",11,['place potatoes in a large pot of lightly sal...,"this is a super easy, great tasting, make ahea...","['spreadable cheese with garlic and herbs', 'n...",11
4,amish tomato ketchup for canning,44061,190,41706,2002-10-25,"['weeknight', 'time-to-make', 'course', 'main-...","[352.9, 1.0, 337.0, 23.0, 3.0, 0.0, 28.0]",5,['mix all ingredients& boil for 2 1 / 2 hours ...,my dh's amish mother raised him on this recipe...,"['tomato juice', 'apple cider vinegar', 'sugar...",8


In [2]:
# Колонки и типы. У recipes важны: id, name, minutes, nutrition, ingredients, n_steps.
# У interactions: recipe_id, user_id, rating, review.
recipes.info()
print("-" * 60)
interactions.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 231637 entries, 0 to 231636
Data columns (total 12 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   name            231636 non-null  object
 1   id              231637 non-null  int64 
 2   minutes         231637 non-null  int64 
 3   contributor_id  231637 non-null  int64 
 4   submitted       231637 non-null  object
 5   tags            231637 non-null  object
 6   nutrition       231637 non-null  object
 7   n_steps         231637 non-null  int64 
 8   steps           231637 non-null  object
 9   description     226658 non-null  object
 10  ingredients     231637 non-null  object
 11  n_ingredients   231637 non-null  int64 
dtypes: int64(5), object(7)
memory usage: 21.2+ MB
------------------------------------------------------------


,user_id,recipe_id,date,rating,review
0,38094,40893,2003-02-17,4,Great with a salad. Cooked on top of stove for...
1,1293707,40893,2011-12-21,5,"So simple, so delicious! Great for chilly fall..."
2,8937,44394,2002-12-01,4,This worked very well and is EASY. I used not...
3,126440,85009,2010-02-27,5,I made the Mexican topping and took it to bunk...
4,57222,85009,2011-10-01,5,"Made the cheddar bacon topping, adding a sprin..."


## Шаг 2. Парсим колонку `nutrition` («польза»)

В Food.com `nutrition` — это строка-список из 7 чисел в фиксированном порядке:
`[калории, жиры %DV, сахар %DV, натрий %DV, белок %DV, насыщ. жиры %DV, углеводы %DV]`.
Разложим её на 7 отдельных колонок (%DV = процент от суточной нормы).

In [3]:
nutr_cols = [
    "calories", "total_fat_pdv", "sugar_pdv", "sodium_pdv",
    "protein_pdv", "sat_fat_pdv", "carbs_pdv",
]

# ast.literal_eval безопасно превращает строку "[1, 2, 3]" в настоящий список
parsed = recipes["nutrition"].apply(ast.literal_eval)
recipes[nutr_cols] = pd.DataFrame(parsed.tolist(), index=recipes.index)

recipes[["name"] + nutr_cols].head()

,name,calories,total_fat_pdv,sugar_pdv,sodium_pdv,protein_pdv,sat_fat_pdv,carbs_pdv
0,arriba baked winter squash mexican style,51.5,0.0,13.0,0.0,2.0,0.0,4.0
1,a bit different breakfast pizza,173.4,18.0,0.0,17.0,22.0,35.0,1.0
2,all in the kitchen chili,269.8,22.0,32.0,48.0,39.0,27.0,5.0
3,alouette potatoes,368.1,17.0,10.0,2.0,14.0,8.0,20.0
4,amish tomato ketchup for canning,352.9,1.0,337.0,23.0,3.0,0.0,28.0


## Шаг 3. Агрегируем оценки и отзывы по рецепту («популярность» + «реакция»)

В `interactions` одна строка = один отзыв пользователя. Свернём их до уровня рецепта.

In [4]:
agg = interactions.groupby("recipe_id").agg(
    n_ratings=("rating", "size"),                       # сколько оценок (популярность)
    avg_rating=("rating", "mean"),                      # средний рейтинг
    n_reviews=("review", lambda s: s.notna().sum()),    # сколько текстовых отзывов
)
print(agg.shape)
agg.sort_values("n_ratings", ascending=False).head()

(231637, 3)


,n_ratings,avg_rating,n_reviews
recipe_id,,,
2886,1613,4.185989,1609
27208,1601,4.288570,1601
89204,1579,4.220393,1579
39087,1448,4.541436,1448
67256,1322,4.329047,1322


## Шаг 4. Собираем мастер-таблицу — три «правды» в одном месте

Соединяем рецепты (польза) с агрегатами (популярность + реакция) по ключу `id == recipe_id`.

In [5]:
df = recipes.merge(agg, left_on="id", right_index=True, how="inner")
print("Мастер-таблица:", df.shape)

# Сохраним для дальнейшего анализа
df.to_csv(DATA / "recipes_master.csv", index=False)

df[["name", "minutes", "calories", "n_ratings", "avg_rating", "n_reviews"]].head()

Мастер-таблица: (231637, 22)


,name,minutes,calories,n_ratings,avg_rating,n_reviews
0,arriba baked winter squash mexican style,55,51.5,3,5.0,3
1,a bit different breakfast pizza,30,173.4,4,3.5,4
2,all in the kitchen chili,130,269.8,1,4.0,1
3,alouette potatoes,45,368.1,2,4.5,2
4,amish tomato ketchup for canning,190,352.9,1,5.0,1


## Что дальше — ищем контрасты

Когда мастер-таблица готова, можно проверять гипотезы:

1. **Популярность ≠ польза:** корреляция `avg_rating` / `n_ratings` с `calories`, `sugar_pdv`, `sat_fat_pdv`.
2. **Надёжность рейтинга:** отфильтровать рецепты с малым числом оценок (напр. `n_ratings >= 10`).
3. **О чём отзывы:** частоты слов / тональность текстов `review` — хвалят ли за вкус/простоту, а не за пользу?
4. **(опц.) USDA:** уточнить нутриенты по ингредиентам.

## Шаг 5. Уточнение нутриентов через USDA FoodData Central (API)

USDA FDC — это **REST API**, а не файл-датасет. Он даёт точные нутриенты на 100 г для обобщённых продуктов. Здесь мы делаем функцию `usda_lookup(ингредиент)`, которая ищет продукт и возвращает калории/белки/жиры/углеводы/сахар/натрий.

In [9]:
import os
import time
from pathlib import Path

import requests

# Ключ читаем из day_2/.env (файл вне git: *.env в .gitignore).
# Формат строки в файле:  USDA_API_KEY=твой_ключ
def load_usda_key():
    if os.environ.get("USDA_API_KEY"):
        return os.environ["USDA_API_KEY"]
    env = Path(".env")
    if env.exists():
        for line in env.read_text().splitlines():
            if line.startswith("USDA_API_KEY="):
                return line.split("=", 1)[1].strip()
    return "DEMO_KEY"   # запасной вариант с жёстким лимитом (~30 запросов/час)

USDA_API_KEY = load_usda_key()
USDA_SEARCH = "https://api.nal.usda.gov/fdc/v1/foods/search"

# id нутриентов USDA -> понятные имена. Значения возвращаются НА 100 Г продукта.
USDA_NUTRIENTS = {
    1008: "energy_kcal", 1003: "protein_g", 1004: "fat_g",
    1005: "carbs_g",     2000: "sugar_g",   1093: "sodium_mg",
}

In [10]:
def usda_lookup(query, api_key=USDA_API_KEY):
    """Ищет продукт в USDA и возвращает нутриенты на 100 г.
    Берём только Foundation/SR Legacy — это обобщённые продукты,
    а не брендовые товары (у тех значения шумные)."""
    params = {
        "query": query,
        "pageSize": 1,
        "dataType": "Foundation,SR Legacy",
        "api_key": api_key,
    }
    r = requests.get(USDA_SEARCH, params=params, timeout=30)
    r.raise_for_status()
    foods = r.json().get("foods", [])
    if not foods:
        return None
    food = foods[0]
    out = {"query": query, "matched": food["description"], "fdcId": food["fdcId"]}
    for n in food["foodNutrients"]:
        if n["nutrientId"] in USDA_NUTRIENTS:
            out[USDA_NUTRIENTS[n["nutrientId"]]] = n["value"]
    return out

usda_lookup("banana")   # проба

{'query': 'banana',
 'matched': 'Bananas, dehydrated, or banana powder',
 'fdcId': 173945,
 'protein_g': 3.89,
 'sodium_mg': 3.0,
 'fat_g': 1.81,
 'carbs_g': 88.3,
 'energy_kcal': 346,
 'sugar_g': 47.3}

In [11]:
# Демо: соберём нутриенты для нескольких ингредиентов в DataFrame.
# time.sleep — вежливость к API и спасение от лимита DEMO_KEY.
sample = ["banana", "butter", "white sugar", "olive oil", "egg"]
rows = []
for ing in sample:
    rows.append(usda_lookup(ing))
    time.sleep(1)

pd.DataFrame(rows)

,query,matched,fdcId,protein_g,sodium_mg,fat_g,carbs_g,energy_kcal,sugar_g
0,banana,"Bananas, dehydrated, or banana powder",173945,3.89,3.0,1.81,88.30,346.0,47.3
1,butter,"Butter, Clarified butter (ghee)",171314,0.00,0.0,100.00,0.00,900.0,0.0
2,white sugar,"Sugar, turbinado",170674,0.00,3.0,0.00,99.80,399.0,99.2
3,olive oil,"Oil, corn, peanut, and olive",167737,0.00,0.0,100.00,0.00,884.0,0.0
4,egg,"Eggs, Grade A, Large, egg white",747997,10.70,NaN,0.00,2.36,55.0,NaN


### Нюансы USDA и как связать с рецептами

- **Качество совпадения.** Поиск по строке неточный: `banana` может найти «банановый порошок», а `chicken breast` — «нарезку для бутербродов» (иногда вообще без нутриентов). Для надёжности: уточняй запрос (`raw banana`), ограничивай `SR Legacy`, или составь ручной словарь для топ-ингредиентов.
- **Лимиты API.** `DEMO_KEY` — ~30 запросов/час. Ингредиентов десятки тысяч, поэтому:
  1. собери **уникальные** ингредиенты из `recipes["ingredients"]`;
  2. запрашивай только их и **кэшируй** ответы (чтобы не дёргать API дважды);
  3. получи свой бесплатный ключ для объёма.
- **Зачем это нужно.** Food.com даёт нутриенты как `%DV` на весь рецепт; USDA даёт точные граммы на 100 г ингредиента — это независимая перепроверка «пользы».